In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(
    r"c:\Users\nandu\Desktop\Socialdata\merged_FULL.csv",
    low_memory=False
)

print("Rows:", len(df))


In [ ]:
import pandas as pd

df = pd.read_csv("merged_FULL.csv", low_memory=False)

# Clean column names (very important)
df.columns = df.columns.str.strip()

print(df.columns)

In [ ]:
print(df.columns)

In [2]:
import pandas as pd

# Load FULL dataset
df = pd.read_csv("merged_FULL.csv", low_memory=False)

# Clean column names
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns)

print("Year range:",
      df['Incident Year'].min(),
      "-",
      df['Incident Year'].max())

# Create small dataset
df_small = (
    df.groupby('Incident Year', group_keys=False)
      .apply(lambda x: x.sample(min(len(x), 300)))
)

df_small.to_csv("merged_small.csv", index=False)

print("Small dataset created successfully.")

Columns:
Index(['Row ID', 'Incident Datetime', 'Incident Date', 'Incident Time',
       'Incident Year', 'Incident Day of Week', 'Report Datetime',
       'Incident ID', 'Incident Number', 'CAD Number', 'Report Type Code',
       'Report Type Description', 'Filed Online', 'Incident Code',
       'Incident Category', 'Incident Subcategory', 'Incident Description',
       'Resolution', 'Intersection', 'CNN', 'Police District',
       'Analysis Neighborhood', 'Supervisor District',
       'Supervisor District 2012', 'Latitude', 'Longitude', 'Point',
       'data_as_of', 'data_loaded_at', 'PdId', 'IncidntNum', 'Category',
       'Descript', 'DayOfWeek', 'Date', 'Time', 'PdDistrict', 'Address', 'X',
       'Y', 'location'],
      dtype='str')
Year range: 2018.0 - 2026.0
Small dataset created successfully.


In [3]:
print(len(df))
print(df['Incident Year'].value_counts().sort_index())

3075676
Incident Year
2018.0    147506
2019.0    143027
2020.0    114601
2021.0    125336
2022.0    133048
2023.0    130932
2024.0    109116
2025.0     93749
2026.0      6625
Name: count, dtype: int64


In [ ]:
df['Incident Date'] = pd.to_datetime(df['Incident Date'], dayfirst=True, errors='coerce')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

df['Combined Date'] = df['Incident Date'].fillna(df['Date'])
df['Year'] = df['Combined Date'].dt.year
print("Full datset year rannge:",df['Year'].min(),"-",df['Year'].max())
#df = df[(df['Year'] >= 2003) & (df['Year'] <= 2025)]

#print("Filtered range:", df['Year'].min(), "-", df['Year'].max())
#print("Min year:", df['Year'].min())
#print("Max year:", df['Year'].max())


In [ ]:
df = df[(df['Year'] >= 2003) & (df['Year'] <= 2025)]

print("Filtered range:", df['Year'].min(), "-", df['Year'].max())

In [ ]:
df['Category_All'] = df['Incident Category'].fillna(df['Category'])

print(df['Category_All'].value_counts().head(20))

In [ ]:
df['Category_All'] = df['Incident Category'].fillna(df['Category'])

In [ ]:
df['Category_All'] = df['Category_All'].str.upper()

In [ ]:
df['Category_All'] = df['Category_All'].replace({
    'LARCENY/THEFT': 'LARCENY THEFT',
    'VEHICLE THEFT': 'MOTOR VEHICLE THEFT'
})

In [ ]:
focus_crimes = [
    'LARCENY THEFT',
    'ASSAULT',
    'BURGLARY',
    'MOTOR VEHICLE THEFT',
    'ROBBERY'
]

In [ ]:
df_focus = df[df['Category_All'].isin(focus_crimes)]

print("df_focus year range:", df_focus['Year'].min(), "-", df_focus['Year'].max())

In [ ]:
yearly_counts = (
    df_focus
    .groupby(['Year', 'Category_All'])
    .size()
    .reset_index(name='Total Incidents')
)
print(f"Year range: {yearly_counts['Year'].min()} - {yearly_counts['Year'].max()}")
#print("Year range:", yearly_counts['Year'].min(), "-", yearly_counts['Year'].max())
yearly_counts.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))

sns.lineplot(
    data=yearly_counts,
    x='Year',
    y='Total Incidents',
    hue='Category_All'
)

plt.title("Total Incidents Per Year (2003–2025)")
plt.xlabel("Year")
plt.ylabel("Number of Incidents")
plt.xticks(range(2003, 2026, 2))
plt.legend(title="Crime Category")
plt.tight_layout()
plt.show()

In [ ]:
df_focus['District_All'] = df_focus['Police District'].fillna(df_focus['PdDistrict'])
df_focus['District_All'] = df_focus['District_All'].str.upper()
print(df_focus['District_All'].unique())

In [ ]:
city_totals = df_focus['Category_All'].value_counts(normalize=True)

print(city_totals)

In [ ]:
district_totals = (
    df_focus
    .groupby('District_All')['Category_All']
    .value_counts(normalize=True)
    .unstack()
)

In [ ]:
ratio = district_totals.div(city_totals, axis=1)

ratio.head()

In [ ]:
import pandas as pd

df = pd.read_csv("merged_FULL.csv")

# keep smaller portion
df_small = df.sample(5000)

df_small.to_csv("merged_small.csv", index=False)

In [ ]:
df = pd.read_csv("merged_FULL.csv", low_memory=False)

In [ ]:
import pandas as pd

df = pd.read_csv("merged_FULL.csv", low_memory=False)

# create smaller sample
df_small = df.sample(5000)

df_small.to_csv("merged_small.csv", index=False)

print("Small dataset created successfully!")

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(ratio, annot=True, cmap='coolwarm', center=1)

plt.title("Conditional Crime Profile by Police District")
plt.xlabel("Crime Type")
plt.ylabel("Police District")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Make sure district names are consistent
ratio.index = ratio.index.str.upper()

ratio.plot(kind='bar', figsize=(12,6))

plt.title("Conditional Crime Profile by Police District")
plt.xlabel("Police District")
plt.ylabel("Relative Ratio (P(crime | district) / P(crime))")
plt.axhline(1, color='black', linestyle='--')  # baseline
plt.legend(title="Crime Type")
plt.tight_layout()
plt.show()